In [ ]:
# قدم 1: extract کن
!unzip frames.zip -d /content/frames

In [ ]:

import cv2
import numpy as np
import json
import os
from pathlib import Path
from google.colab.patches import cv2_imshow

# ═══════════════════════════════════════════════════════════════
# 0. LOAD FRAMES
# ═══════════════════════════════════════════════════════════════
frames_dir = Path("/content/frames")          # ← مسیر فریم‌هات

frame_files = sorted(frames_dir.glob("*.jpg"))
if not frame_files:
    frame_files = sorted(frames_dir.glob("**/*.jpg"))
if not frame_files:
    frame_files = sorted(frames_dir.glob("*.png"))

assert len(frame_files) > 0, f"No frames found in {frames_dir}"
print(f"✅ {len(frame_files)} frames found")

first = cv2.imread(str(frame_files[0]))
h_img, w_img = first.shape[:2]
print(f"   Resolution: {w_img}×{h_img}")

# ═══════════════════════════════════════════════════════════════
# 1. PARAMS  ← اگه دوربین فرق داره اینا رو تنظیم کن
# ═══════════════════════════════════════════════════════════════
CONV_Y   = int(h_img * 0.57)   # بالای ناحیه conveyor
CONV_X   = int(w_img * 0.52)   # چپ ناحیه conveyor
ROI_X    = int(w_img * 0.60)   # ناحیه مهم پایین-راست
ROI_Y    = int(h_img * 0.60)
FLOW_THR = 0.30                 # آستانه فلو (کمتر = حساس‌تر)
MIN_AREA = 500                  # حداقل مساحت سنگ (px²)
SHOW_N   = 12                   # تعداد فریم در موزائیک

# kernels
ko3  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,  3 ))
kc16 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (16, 16))
kc24 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (24, 24))
kc35 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (35, 35))

# ═══════════════════════════════════════════════════════════════
# 2. BUILD REFERENCE BACKGROUND (median of all frames)
# ═══════════════════════════════════════════════════════════════
print("⚙ Building background model …")
stack = []
for fpath in frame_files[::3]:          # هر ۳ فریم یک بار (سریع‌تر)
    g = cv2.GaussianBlur(
        cv2.cvtColor(cv2.imread(str(fpath)), cv2.COLOR_BGR2GRAY),
        (5, 5), 1.5)
    stack.append(g.astype(np.float32))

bg_gray = np.median(np.array(stack), axis=0).astype(np.uint8)
bg_std  = np.std(np.array(stack),   axis=0).astype(np.float32)
print(f"   Background ready — dynamic pixels: {(bg_std>8).sum()}")

# ═══════════════════════════════════════════════════════════════
# 3. PRE-COMPUTE OPTICAL FLOW MAGNITUDES
# ═══════════════════════════════════════════════════════════════
print("⚙ Computing optical flow …")
flow_mags = []
for i in range(len(frame_files)):
    g1 = cv2.GaussianBlur(
         cv2.cvtColor(cv2.imread(str(frame_files[max(0, i-1)])),
                      cv2.COLOR_BGR2GRAY), (5,5), 1.0)
    g2 = cv2.GaussianBlur(
         cv2.cvtColor(cv2.imread(str(frame_files[i])),
                      cv2.COLOR_BGR2GRAY), (5,5), 1.0)
    flow = cv2.calcOpticalFlowFarneback(
               g1, g2, None, 0.5, 5, 15, 5, 7, 1.5, 0)
    mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    flow_mags.append(mag)
    if i % 40 == 0:
        print(f"   {i}/{len(frame_files)}")

print("✅ Flow computed")

# ═══════════════════════════════════════════════════════════════
# 4. DETECT STONE — core function
# ═══════════════════════════════════════════════════════════════
def detect_stone(idx):
    """
    Returns (annotated_frame, stone_dict or None)

    stone_dict:
        bbox    : [x, y, w, h]
        minRect : [width, height, angle_deg]
        center  : [cx, cy]
        area    : float  (px²)
        solidity: float  (0-1)
        in_roi  : bool
    """
    frame = cv2.imread(str(frame_files[idx]))
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # ── 4a. Temporal flow average ±2 frames ───────────────────
    lo = max(0, idx-2)
    hi = min(len(frame_files)-1, idx+2)
    avg_mag = np.mean(flow_mags[lo:hi+1], axis=0)

    fm = (avg_mag > FLOW_THR).astype(np.uint8) * 255
    fm[:CONV_Y, :] = 0
    fm[:, :CONV_X] = 0
    fm = cv2.morphologyEx(fm, cv2.MORPH_OPEN, ko3)

    if fm.sum() // 255 < MIN_AREA:
        return frame, None

    # ── 4b. Closing — 3 passes to fill roller gaps + dilate ───
    fm2 = cv2.morphologyEx(fm,  cv2.MORPH_CLOSE, kc16)
    fm2 = cv2.morphologyEx(fm2, cv2.MORPH_CLOSE, kc24)
    fm2 = cv2.morphologyEx(fm2, cv2.MORPH_CLOSE, kc35)   # fill centre
    fm2 = cv2.dilate(fm2, kc24, iterations=1)             # expand edges

    if fm2.sum() // 255 < MIN_AREA:
        return frame, None

    # ── 4c. Z-score guide from background model ───────────────
    diff  = cv2.absdiff(gray.astype(np.float32), bg_gray.astype(np.float32))
    zsc   = diff / (bg_std + 2.0)
    zsc[fm2 == 0] = 0                    # only inside flow zone
    zsc_mask = (zsc > 1.5).astype(np.uint8) * 255
    zsc_mask = cv2.morphologyEx(zsc_mask, cv2.MORPH_CLOSE, kc24)

    # combine: flow zone AND z-score agree
    combined = cv2.bitwise_and(fm2, zsc_mask)
    combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kc35)
    combined = cv2.dilate(combined, kc24, iterations=1)

    # fallback: if combined is empty, use flow zone alone
    if combined.sum() // 255 < MIN_AREA:
        combined = fm2.copy()

    # ── 4d. Pick best contour ─────────────────────────────────
    cnts, _ = cv2.findContours(combined, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
    best, best_score = None, 0
    for cnt in cnts:
        area = cv2.contourArea(cnt)
        if area < MIN_AREA:
            continue
        x, y, bw, bh = cv2.boundingRect(cnt)
        if y+bh < CONV_Y or x+bw < CONV_X:
            continue
        ar = bw / (bh + 1e-5)
        if ar < 0.10 or ar > 9:          # filter extreme shapes
            continue
        sol = area / (cv2.contourArea(cv2.convexHull(cnt)) + 1e-5)
        cm  = np.zeros((h_img, w_img), np.uint8)
        cv2.drawContours(cm, [cnt], -1, 255, -1)
        ov  = float(cv2.bitwise_and(cm, fm).sum()) / 255
        score = area * sol * (ov / (area + 1))
        if score > best_score:
            best_score, best = score, cnt

    if best is None:
        return frame, None

    # ── 4e. Convex hull → clean shape ─────────────────────────
    hull = cv2.convexHull(best)
    hm   = np.zeros((h_img, w_img), np.uint8)
    cv2.drawContours(hm, [hull], -1, 255, -1)
    hm = cv2.morphologyEx(hm, cv2.MORPH_CLOSE,
                          cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9)))
    cnts2, _ = cv2.findContours(hm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts2:
        return frame, None
    final_cnt = max(cnts2, key=cv2.contourArea)

    # ── 4f. Measurements ──────────────────────────────────────
    rect    = cv2.minAreaRect(final_cnt)
    box     = np.intp(cv2.boxPoints(rect))
    rw, rh  = int(rect[1][0]), int(rect[1][1])
    ang     = float(rect[2])
    cx, cy  = int(rect[0][0]), int(rect[0][1])
    x, y, bw, bh = cv2.boundingRect(final_cnt)
    area    = float(cv2.contourArea(final_cnt))
    sol     = area / (cv2.contourArea(cv2.convexHull(final_cnt)) + 1e-5)
    in_roi  = (x+bw > ROI_X) and (y+bh > ROI_Y)

    stone = dict(
        bbox    = [x, y, bw, bh],
        minRect = [rw, rh, ang],
        center  = [cx, cy],
        area    = area,
        solidity= float(sol),
        in_roi  = in_roi
    )

    # ── 4g. Annotate frame ────────────────────────────────────
    vis = frame.copy()

    # ROI zone
    ov2 = vis.copy()
    cv2.rectangle(ov2, (ROI_X, ROI_Y), (w_img-1, h_img-1), (0,180,255), -1)
    cv2.addWeighted(ov2, 0.07, vis, 0.93, 0, vis)
    cv2.rectangle(vis, (ROI_X, ROI_Y), (w_img-1, h_img-1), (0,180,255), 2)
    cv2.putText(vis, "ROI", (ROI_X+5, ROI_Y+18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,180,255), 1)

    # filled stone overlay
    ov3 = vis.copy()
    cv2.drawContours(ov3, [final_cnt], -1, (0,55,210), -1)
    cv2.addWeighted(ov3, 0.30, vis, 0.70, 0, vis)

    # cyan contour
    cv2.drawContours(vis, [final_cnt], -1, (0,230,255), 2)

    # green rotated rect
    cv2.drawContours(vis, [box], 0, (0,255,70), 2)

    # yellow axis-aligned corner ticks
    tick = 9
    for px, py in [(x,y),(x+bw,y),(x,y+bh),(x+bw,y+bh)]:
        dx = tick if px==x else -tick
        dy = tick if py==y else -tick
        cv2.line(vis,(px,py),(px+dx,py),(255,220,0),2)
        cv2.line(vis,(px,py),(px,py+dy),(255,220,0),2)

    # width arrow
    ay = max(y-18, 15)
    cv2.arrowedLine(vis,(x,ay),(x+bw,ay),(255,200,0),1,tipLength=0.08)
    cv2.arrowedLine(vis,(x+bw,ay),(x,ay),(255,200,0),1,tipLength=0.08)
    cv2.putText(vis, f"{bw}px", (x+bw//2-15, ay-4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255,200,0), 1)

    # height arrow
    ax2 = min(x+bw+16, w_img-40)
    cv2.arrowedLine(vis,(ax2,y),(ax2,y+bh),(255,200,0),1,tipLength=0.08)
    cv2.arrowedLine(vis,(ax2,y+bh),(ax2,y),(255,200,0),1,tipLength=0.08)
    cv2.putText(vis, f"{bh}px", (ax2+3, y+bh//2+5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255,200,0), 1)

    # minRect label
    cv2.putText(vis, f"{rw}x{rh}px  {ang:.1f}deg",
                (box[1][0]+4, box[1][1]+14),
                cv2.FONT_HERSHEY_SIMPLEX, 0.36, (0,255,70), 1)

    # center crosshair
    cv2.line(vis,(cx-14,cy),(cx+14,cy),(0,255,70),1)
    cv2.line(vis,(cx,cy-14),(cx,cy+14),(0,255,70),1)
    cv2.circle(vis,(cx,cy),6,(0,255,70),-1)
    cv2.circle(vis,(cx,cy),6,(255,255,255),1)

    # info panel
    loc   = "ROI" if in_roi else "zone"
    panel = [
        f"STONE [{loc}]",
        f"BBox  {bw} x {bh} px",
        f"minRect {rw} x {rh} px",
        f"Angle   {ang:.1f} deg",
        f"Area    {int(area):,} px2",
        f"Solidity {sol:.2f}",
        f"Center  ({cx},{cy})",
    ]
    pxw=170; lhp=16; pad=5
    pxh = len(panel)*lhp + pad*2
    px0 = min(x, w_img-pxw-4)
    py0 = max(0, y-8-pxh)
    cv2.rectangle(vis,(px0,py0),(px0+pxw,py0+pxh),(12,12,12),-1)
    cv2.rectangle(vis,(px0,py0),(px0+pxw,py0+pxh),(0,255,70),1)
    for li, ln in enumerate(panel):
        c = (0,255,80) if li==0 \
            else (180,255,200) if ("minRect" in ln or "Angle" in ln) \
            else (215,215,215)
        cv2.putText(vis, ln, (px0+4, py0+pad+(li+1)*lhp-2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.36, c, 1)

    # status bar
    bar = (f"Frame {idx:03d}/{len(frame_files)-1}  |  "
           f"Objects:1  |  ROI:{int(in_roi)}  |  "
           f"{rw}x{rh}px  Angle:{ang:.1f}deg  Sol:{sol:.2f}  |  "
           f"CYAN=precise contour  GREEN=minAreaRect  YELLOW=bbox")
    cv2.rectangle(vis,(0,0),(w_img,30),(10,10,10),-1)
    cv2.putText(vis, bar, (5,21),
                cv2.FONT_HERSHEY_SIMPLEX, 0.40, (180,230,200), 1)

    return vis, stone


# ═══════════════════════════════════════════════════════════════
# 5. RUN ALL FRAMES
# ═══════════════════════════════════════════════════════════════
print("\n🔍 Detecting stones in all frames …")
results  = []
key_imgs = []

for idx in range(len(frame_files)):
    vis, info = detect_stone(idx)
    results.append({"frame": idx, "stone": info})

    if info and len(key_imgs) < SHOW_N:
        thumb = cv2.resize(vis, (424, 263))
        cv2.putText(thumb, f"f{idx}", (5, 255),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
        key_imgs.append(thumb)

    if idx % 20 == 0:
        status = "STONE" if info else "---"
        print(f"   Frame {idx:03d}: {status}")

print("✅ Done")

# ═══════════════════════════════════════════════════════════════
# 6. MOSAIC
# ═══════════════════════════════════════════════════════════════
cols = 4
while len(key_imgs) % cols:
    key_imgs.append(np.zeros((263, 424, 3), np.uint8))

rows   = [np.hstack(key_imgs[i:i+cols]) for i in range(0, len(key_imgs), cols)]
mosaic = np.vstack(rows)

hdr = np.full((50, mosaic.shape[1], 3), (12,12,12), np.uint8)
cv2.putText(hdr,
    "STONE DETECTION | Flow + ZScore + minAreaRect | "
    "CYAN=contour  GREEN=minRect(rotated)  YELLOW=bbox",
    (8, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.46, (0,200,255), 1)

print("\n📊 Mosaic of detected frames:")
cv2_imshow(np.vstack([hdr, mosaic]))

# ═══════════════════════════════════════════════════════════════
# 7. TRAJECTORY MAP
# ═══════════════════════════════════════════════════════════════
mid = cv2.imread(str(frame_files[len(frame_files)//2]))
canvas = mid.copy()
dark = canvas.copy(); dark[:] = 15
cv2.addWeighted(dark, 0.45, canvas, 0.55, 0, canvas)
cv2.rectangle(canvas,(ROI_X,ROI_Y),(w_img-1,h_img-1),(0,180,255),2)

traj = [(r["frame"], r["stone"]["center"])
        for r in results if r["stone"] and r["stone"]["in_roi"]]

for i in range(1, len(traj)):
    frac = i / max(len(traj)-1, 1)
    c = (0, int(255*(1-frac)), int(200*frac))
    cv2.line(canvas, traj[i-1][1], traj[i][1], c, 2, cv2.LINE_AA)

if traj:
    cv2.circle(canvas, traj[0][1],  10, (0,255,0), -1)
    cv2.circle(canvas, traj[-1][1], 10, (0,0,255), -1)
    cv2.putText(canvas, "START", (traj[0][1][0]+12,  traj[0][1][1]+4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
    cv2.putText(canvas, "END",   (traj[-1][1][0]+12, traj[-1][1][1]+4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)

cv2.rectangle(canvas,(0,0),(w_img,30),(10,10,10),-1)
cv2.putText(canvas,
    f"Trajectory | {len(traj)} ROI detections | green to red = time",
    (5,21), cv2.FONT_HERSHEY_SIMPLEX, 0.46, (180,230,200), 1)

print("\n📍 Trajectory map:")
cv2_imshow(canvas)

# ═══════════════════════════════════════════════════════════════
# 8. STATS
# ═══════════════════════════════════════════════════════════════
active = [r["stone"] for r in results if r["stone"]]
ws     = [s["minRect"][0] for s in active]
hs     = [s["minRect"][1] for s in active]
angs   = [s["minRect"][2] for s in active]
areas  = [s["area"]       for s in active]

print(f"\n{'─'*55}")
print(f"  Total frames      : {len(results)}")
print(f"  Frames with stone : {len(active)}")
print(f"  ROI detections    : {len(traj)}")
print(f"  minRect W  avg={np.mean(ws):.0f}  range={min(ws):.0f}–{max(ws):.0f} px")
print(f"  minRect H  avg={np.mean(hs):.0f}  range={min(hs):.0f}–{max(hs):.0f} px")
print(f"  Angle      avg={np.mean(angs):.1f}  range={min(angs):.1f}–{max(angs):.1f} deg")
print(f"  Area       avg={np.mean(areas):.0f}  range={int(min(areas))}–{int(max(areas))} px2")
print(f"{'─'*55}")

# ═══════════════════════════════════════════════════════════════
# 9. SAVE JSON LOG
# ═══════════════════════════════════════════════════════════════
log_path = "/content/stone_log.json"
with open(log_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\n💾 Log saved: {log_path}")